# RAG Engineering — Document Q&A System with Hybrid Search

## Assignment Objective

Build a complete Retrieval-Augmented Generation (RAG) pipeline that combines:
- RecursiveCharacterTextSplitter for chunking
- `all-MiniLM-L6-v2` for vector embeddings
- ChromaDB for vector storage
- BM25 for keyword retrieval
- Reciprocal Rank Fusion (RRF) for hybrid retrieval
- `cross-encoder/ms-marco-MiniLM-L-6-v2` for reranking
- LangChain + Ollama for grounded answer generation

**Important:** Chunking and embedding are separate steps. `RecursiveCharacterTextSplitter` performs chunking; `all-MiniLM-L6-v2` creates embeddings.

## Pipeline Architecture

```text
Document
   ↓
Chunking (300, overlap 50)
   ↓
 ┌───────────────────────┐
 │                       │
 ↓                       ↓
Vector Search           BM25 Search
 │                       │
 └───────────┬───────────┘
             ↓
      Reciprocal Rank Fusion
             ↓
          Top 10
             ↓
       Cross-Encoder
             ↓
           Top 3
             ↓
       LLM + Context
             ↓
          Answer
```


In [ ]:
# Install dependencies
# Run this cell once if the packages are not installed.

%pip install -q langchain langchain-community langchain-text-splitters langchain-ollama
%pip install -q sentence-transformers chromadb rank-bm25 numpy

In [ ]:
import os
import re
import shutil
import numpy as np

import chromadb
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

print('Libraries imported successfully.')

## 1. Load the Document

Place a knowledge-heavy document of at least two pages at `data/document.txt`.

For this submission, the notebook uses a local text file so the document can be included in the GitHub repository without depending on a live website.

In [ ]:
DATA_DIR = 'data'
DOCUMENT_PATH = os.path.join(DATA_DIR, 'document.txt')

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(DOCUMENT_PATH):
    raise FileNotFoundError(
        f'Missing {DOCUMENT_PATH}. Add a 2+ page knowledge-heavy text document before running this cell.'
    )

with open(DOCUMENT_PATH, 'r', encoding='utf-8') as f:
    document_text = f.read()

print('Document characters:', len(document_text))
print('\nFirst 1,000 characters:\n')
print(document_text[:1000])

## 2. Split the Document into Chunks

Assignment settings:
- `chunk_size = 300`
- `chunk_overlap = 50`

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = text_splitter.split_text(document_text)

print('Number of chunks:', len(chunks))
print('\nFirst 5 chunks:')

for i, chunk in enumerate(chunks[:5]):
    print(f'\n--- Chunk {i} ---')
    print(chunk)

## 3. Create Vector Embeddings

The embedding model converts each chunk into a numerical vector. `all-MiniLM-L6-v2` produces compact semantic embeddings suitable for similarity search.

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

chunk_embeddings = embedding_model.encode(
    chunks,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

print('Embedding matrix shape:', chunk_embeddings.shape)
print('Embedding dimension:', chunk_embeddings.shape[1])

## 4. Store Embeddings in ChromaDB

In [ ]:
CHROMA_PATH = './chroma_db'
COLLECTION_NAME = 'document_qa'

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# Recreate the collection so rerunning the notebook does not create duplicate IDs.
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass

collection = chroma_client.create_collection(
    name=COLLECTION_NAME,
    metadata={'description': 'RAG assignment document chunks'}
)

ids = [f'chunk_{i}' for i in range(len(chunks))]

collection.add(
    ids=ids,
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    metadatas=[{'chunk_id': i} for i in range(len(chunks))]
)

print('Chunks stored in ChromaDB:', collection.count())

## 5. Build the BM25 Index

BM25 is a keyword-based retrieval method. It is especially useful when a question contains exact or rare terms that appear in the document.

In [ ]:
def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

tokenized_chunks = [tokenize(chunk) for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print('BM25 index created for', len(tokenized_chunks), 'chunks.')

## 6. Vector Search

In [ ]:
def vector_search(query, top_k=10):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    result = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=min(top_k, len(chunks)),
        include=['documents', 'distances']
    )

    return [
        (chunk_id, text, distance)
        for chunk_id, text, distance in zip(
            result['ids'][0],
            result['documents'][0],
            result['distances'][0]
        )
    ]

vector_results = vector_search('What is the main topic of the document?', top_k=5)
for rank, (chunk_id, text, distance) in enumerate(vector_results, 1):
    print(f'Rank {rank} | {chunk_id} | distance={distance:.4f}')
    print(text[:250], '\n')

## 7. BM25 Search

In [ ]:
def bm25_search(query, top_k=10):
    query_tokens = tokenize(query)
    scores = bm25.get_scores(query_tokens)
    ranked_indices = np.argsort(scores)[::-1][:min(top_k, len(chunks))]

    return [
        (f'chunk_{idx}', chunks[idx], float(scores[idx]))
        for idx in ranked_indices
    ]

bm25_results = bm25_search('exact keyword', top_k=5)
for rank, (chunk_id, text, score) in enumerate(bm25_results, 1):
    print(f'Rank {rank} | {chunk_id} | BM25 score={score:.4f}')
    print(text[:250], '\n')

## 8. Reciprocal Rank Fusion (RRF)

RRF combines ranked results from vector search and BM25 without requiring their raw scores to be on the same numerical scale.

Formula:

`RRF score = sum(1 / (k + rank))`

This implementation uses `k = 60`.

In [ ]:
def reciprocal_rank_fusion(vector_results, bm25_results, k=60, top_k=10):
    scores = {}
    documents = {}

    for rank, (chunk_id, text, _) in enumerate(vector_results, start=1):
        scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)
        documents[chunk_id] = text

    for rank, (chunk_id, text, _) in enumerate(bm25_results, start=1):
        scores[chunk_id] = scores.get(chunk_id, 0.0) + 1.0 / (k + rank)
        documents[chunk_id] = text

    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)

    return [
        {
            'id': chunk_id,
            'text': documents[chunk_id],
            'rrf_score': score
        }
        for chunk_id, score in ranked[:top_k]
    ]

## 9. Hybrid Search

The required function is `hybrid_search(query, top_k=10)`. It retrieves candidates from both systems and combines their rankings using RRF.

In [ ]:
def hybrid_search(query, top_k=10):
    vector_results = vector_search(query, top_k=top_k)
    bm25_results = bm25_search(query, top_k=top_k)

    return reciprocal_rank_fusion(
        vector_results,
        bm25_results,
        k=60,
        top_k=top_k
    )

query = 'What is the main topic of the document?'
hybrid_results = hybrid_search(query, top_k=10)

for rank, result in enumerate(hybrid_results, 1):
    print(f'Rank {rank} | {result["id"]} | RRF={result["rrf_score"]:.6f}')
    print(result['text'][:300], '\n')

## 10. Cross-Encoder Reranking

The cross-encoder reads the question and each retrieved chunk together and assigns a relevance score. We rerank the top 10 hybrid candidates and keep the best 3.

In [ ]:
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank_results(query, results, top_k=3):
    pairs = [(query, result['text']) for result in results]
    scores = reranker.predict(pairs)

    reranked = []
    for result, score in zip(results, scores):
        item = result.copy()
        item['rerank_score'] = float(score)
        reranked.append(item)

    reranked.sort(key=lambda item: item['rerank_score'], reverse=True)
    return reranked[:top_k]

final_retrieved = rerank_results(query, hybrid_results, top_k=3)

for rank, result in enumerate(final_retrieved, 1):
    print(f'Final Rank {rank} | {result["id"]} | rerank={result["rerank_score"]:.4f}')
    print(result['text'], '\n')

## 11. Configure the LLM

This notebook uses a local Ollama model. Make sure Ollama is installed and the selected model has been pulled before running this section.

Example terminal command:

```bash
ollama pull qwen3:1.7b
```

If a different local model is installed, change `OLLAMA_MODEL` below.

In [ ]:
OLLAMA_MODEL = 'qwen3:1.7b'

llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0
)

print('LLM configured:', OLLAMA_MODEL)

## 12. RAG System Prompt

Required instruction:

> Answer using ONLY the context. If not found, say `Not in context.`

In [ ]:
SYSTEM_PROMPT = """
You are a document question-answering assistant.

Answer using ONLY the context provided below.

If the answer cannot be found in the context, say exactly:
Not in context.

Do not use outside knowledge.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', '{question}')
])

print(SYSTEM_PROMPT)

## 13. Complete RAG Pipeline

Flow: query → hybrid top 10 → cross-encoder top 3 → context → LLM.

In [ ]:
def retrieve_documents(query):
    hybrid_results = hybrid_search(query, top_k=10)
    return rerank_results(query, hybrid_results, top_k=3)


def answer_question(question, verbose=True):
    results = retrieve_documents(question)
    context = '\n\n'.join(result['text'] for result in results)

    messages = prompt.format_messages(
        context=context,
        question=question
    )

    response = llm.invoke(messages)

    if verbose:
        print('QUESTION:\n', question)
        print('\nANSWER:\n', response.content)
        print('\nTOP 3 RETRIEVED CHUNKS:')
        for i, result in enumerate(results, 1):
            print(f'\n--- Source {i} | {result["id"]} | score={result["rerank_score"]:.4f} ---')
            print(result['text'])

    return response.content, results

## 14. Test Question 1 — Basic Document Understanding

Replace this question with one that is clearly answered by your document.

In [ ]:
q1 = 'What is the main topic discussed in the document?'
answer_question(q1)

## 15. Test Question 2 — Exact Keyword Matching (BM25 Strength)

Choose a distinctive term that appears exactly in your document. This test demonstrates why keyword retrieval is useful.

In [ ]:
# Replace the phrase with an exact/rare keyword from your document.
q2 = 'What does the document say about the exact keyword Artificial Intelligence?'
answer_question(q2)

## 16. Test Question 3 — Semantic Understanding (Vector Strength)

Use wording that expresses the idea in a different way from the document. Vector embeddings should help retrieve semantically related chunks.

In [ ]:
q3 = 'How can the concepts described in the document help machines perform tasks that normally require human intelligence?'
answer_question(q3)

## 17. Test Question 4 — Multi-Concept Question

Ask a question that requires combining information from relevant passages.

In [ ]:
q4 = 'What are the main methods, uses, or benefits described in the document?'
answer_question(q4)

## 18. Test Question 5 — Not in Context

This should ask for information that is genuinely absent from your chosen document. The expected grounded behavior is `Not in context.`

In [ ]:
q5 = 'What is the population of Mars according to this document?'
answer_question(q5)

## 19. Run All Five Tests


In [ ]:
test_questions = [q1, q2, q3, q4, q5]

for i, question in enumerate(test_questions, 1):
    print('\n' + '=' * 90)
    print(f'TEST QUESTION {i}: {question}')
    answer, _ = answer_question(question, verbose=False)
    print('ANSWER:', answer)

## 20. Edge Case Awareness

**Potential failure point:** malformed or missing document input. If `data/document.txt` is missing or empty, the pipeline cannot create useful chunks or retrieval indexes. The implementation checks for a missing file and raises a clear error. In production, I would also validate that the document contains enough meaningful text before indexing it.

Other possible failures include:
- Ollama is not running or the selected model is unavailable.
- The embedding or reranker model cannot be downloaded.
- A query contains no useful terms for BM25.
- The document does not contain the requested information.
- Too few chunks exist to request the desired top-k results.

The implementation handles the last case by limiting retrieval to the number of available chunks.

## 21. Why I Chose This Approach

This project solves a practical document-question-answering problem: a user can ask questions about a private or specialized document without manually searching through every page. Hybrid retrieval is useful because BM25 is strong for exact keywords while vector search is strong for semantic meaning. RRF combines both retrieval signals, and the cross-encoder provides a second relevance check before the context reaches the LLM. This reduces the amount of irrelevant information passed to the model and makes the final answer more grounded in the source document.

A real business example would be an internal company knowledge assistant that answers questions about product manuals, policies, training documents, pricing information, or operating procedures.

## 22. Conclusion

The completed pipeline demonstrates all required RAG engineering stages:

1. Document loading
2. Chunking with 300-character chunks and 50-character overlap
3. Sentence Transformer embeddings using `all-MiniLM-L6-v2`
4. ChromaDB vector storage
5. BM25 keyword retrieval
6. Reciprocal Rank Fusion
7. Cross-encoder reranking to the best 3 chunks
8. LangChain prompt-based LLM generation
9. Context-only answering with a `Not in context.` fallback
10. Five-question evaluation including keyword and semantic retrieval cases